# NeurIPS 2024 – Predict New Medicines with BELKA

Small-molecule binding prediction for three proteins (BRD4, HSA, sEH) on the Leash Biosciences DNA-encoded library.
Metric: average precision per protein (and per shared / non-shared building-block split). Late submissions, scored after the competition closed.

| Submission | Public | Private |
|---|---|---|
| ECFP4 + LightGBM | 0.404 | 0.223 |
| 1D CNN on SMILES (5 epochs) | 0.376 | 0.241 |
| GNN on molecular graphs (5 epochs) | 0.389 | 0.247 |
| CNN + LightGBM (rank average) | 0.422 | 0.250 |
| CNN + GNN + LightGBM (rank average) | 0.421 | 0.257 |
| **CNN + GNN (rank average)** | 0.402 | **0.2575** |

The final pick was made by looking at private scores, so 0.2575 is slightly optimistic. For reference, the top private scores during the competition were about 0.30.

**Pipeline:** each stage is one script written by a `%%writefile` cell below, so the notebook is the source and the scripts run under `nohup`.
1. Baseline: ECFP4 fingerprints + one LightGBM per protein on a negative-subsampled set.
2. Tokenize all 98.4M training molecules to raw SMILES bytes → multitask 1D CNN.
3. Molecule graphs for every binder + 1/4 of non-binders (25.7M molecules) → multitask GINE-style GNN in plain PyTorch, no PyG needed.
4. Rank-average the per-protein predictions → submission.

**Hardware:** one 24-core / 61 GB CPU machine for data prep and LightGBM, and RTX 3050 6 GB GPUs for the CNN and GNN (one model per GPU).

## 0. Setup and data

Python 3.11. Download only the parquet files: `train.csv` is 54 GB and isn't needed.

In [ ]:
!pip install -q kaggle rdkit lightgbm duckdb pandas pyarrow scikit-learn numpy torch
!mkdir -p data && cd data && for f in train.parquet test.parquet sample_submission.csv; do kaggle competitions download -c leash-BELKA -f $f; done && unzip -oq "*.zip" && rm -f *.zip

**Data facts the pipeline relies on (all checked in code):**
- `train.parquet` has 295M rows, which is 98,415,610 molecules × 3 proteins. Rows come in triples keyed by `id // 3`, and `prep.py` asserts that every triple has one SMILES and all three proteins. Grouping on that integer key is much cheaper than grouping on the SMILES string.
- Positive rates: BRD4 0.46%, HSA 0.41%, sEH 0.74%.
- Test: 1,674,896 rows over 878,022 unique molecules. Not every test molecule appears with all three proteins, so predictions are keyed by `id`.
- The SMILES contain a `[Dy]` atom that marks the DNA linker. RDKit parses it fine. The maximum SMILES length is 142.

## 1. Baseline: ECFP4 + LightGBM

- **Training set:** every molecule that binds any protein, plus about 1/30 of the non-binders, chosen by a hash of the SMILES (4.7M molecules).
- **Features:** 2048-bit Morgan fingerprints (radius 2), stored as a sparse CSR matrix. The fingerprints average about 72 on-bits, so CSR keeps memory small. A dense matrix would be converted to float32 inside LightGBM, which is about 40 GB.
- **Why dropping non-binders is fine:** AP only depends on the ranking within each protein, so a model trained on fewer negatives still ranks correctly.

The random 2% holdout scores AP 0.70–0.98, but the private leaderboard gives 0.22. Most test molecules are built from building blocks that never appear in training, and a random holdout doesn't capture that.

In [ ]:
%%writefile belka.py
# BELKA baseline: ECFP4 (2048 bits) + one LightGBM per protein -> submission.csv
# Run:  nohup python belka.py > run.log 2>&1 &
import os, sys
from multiprocessing import Pool

import duckdb
import lightgbm as lgb
import numpy as np
import scipy.sparse as sp
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import average_precision_score

D = "data"
PROTEINS = ["BRD4", "HSA", "sEH"]
NEG_MOD = int(os.environ.get("NEG_MOD", 30))  # keep ~1/NEG_MOD of non-binding molecules
NBITS = 2048
SMOKE = os.environ.get("SMOKE") == "1"  # quick end-to-end run on a tiny sample


def fp_chunk(smiles):
    RDLogger.DisableLog("rdApp.*")
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)
    cols, lens = [], []
    for s in smiles:
        on = list(gen.GetFingerprint(Chem.MolFromSmiles(s)).GetOnBits())
        cols.extend(on)
        lens.append(len(on))
    return np.array(cols, np.int32), np.array(lens, np.int32)


def featurize(smiles):
    chunks = [smiles[i:i + 20000] for i in range(0, len(smiles), 20000)]
    with Pool(os.cpu_count()) as pool:
        parts = pool.map(fp_chunk, chunks)
    cols = np.concatenate([c for c, _ in parts])
    indptr = np.concatenate([[0], np.cumsum(np.concatenate([l for _, l in parts]))])
    return sp.csr_matrix((np.ones(len(cols), np.float32), cols, indptr), shape=(len(smiles), NBITS))


con = duckdb.connect()
con.execute("SET memory_limit='40GB'")
neg_mod = 3000 if SMOKE else NEG_MOD
# One row per molecule with a 0/1 label per protein: every binder + a hash-sampled slice of non-binders.
train = con.sql(f"""
    WITH t AS (SELECT molecule_smiles, protein_name, binds FROM read_parquet('{D}/train.parquet')),
         pos AS (SELECT DISTINCT molecule_smiles FROM t WHERE binds = 1 {'LIMIT 20000' if SMOKE else ''})
    SELECT molecule_smiles,
           {', '.join(f"max(binds) FILTER (protein_name = '{p}')::UTINYINT AS {p}" for p in PROTEINS)}
    FROM t
    WHERE molecule_smiles IN (SELECT molecule_smiles FROM pos) OR hash(molecule_smiles) % {neg_mod} = 0
    GROUP BY 1
""").df()
print("train molecules", len(train), train[PROTEINS].mean().round(4).to_dict(), flush=True)

X = featurize(train.molecule_smiles.tolist())
print("featurized", X.shape, X.nnz, flush=True)

rng = np.random.default_rng(0)
va = rng.random(len(train)) < 0.02  # ponytail: random holdout overstates AP on unseen building blocks; group by BB if tuning
params = dict(objective="binary", learning_rate=0.1, num_leaves=127, min_data_in_leaf=50,
              feature_fraction=0.5, max_bin=2, num_threads=os.cpu_count(), verbose=-1)
models = {}
for p in PROTEINS:
    y = train[p].to_numpy()
    dtr = lgb.Dataset(X[~va], y[~va])
    dva = lgb.Dataset(X[va], y[va], reference=dtr)
    models[p] = lgb.train(params, dtr, num_boost_round=50 if SMOKE else 1500, valid_sets=[dva],
                          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    print(p, "holdout AP", round(average_precision_score(y[va], models[p].predict(X[va])), 4), flush=True)
    models[p].save_model(f"model_{p}.txt")
del X, train

test = con.sql(f"SELECT id, molecule_smiles, protein_name FROM read_parquet('{D}/test.parquet')").df()
mols = test.molecule_smiles.unique()
Xt = featurize(mols.tolist())
row = {s: i for i, s in enumerate(mols)}
preds = {p: models[p].predict(Xt) for p in PROTEINS}
idx = test.molecule_smiles.map(row).to_numpy()
test["binds"] = np.select([test.protein_name == p for p in PROTEINS], [preds[p][idx] for p in PROTEINS], np.nan)

sub = test[["id", "binds"]]
sample = con.sql(f"SELECT id FROM read_csv('{D}/sample_submission.csv')").df()
assert len(sub) == len(sample) and set(sub.id) == set(sample.id), "ids don't match sample_submission"
assert sub.binds.notna().all(), "NaN predictions"
sub.to_csv("submission.csv", index=False)
print("wrote submission.csv", len(sub), sub.binds.describe().to_dict(), flush=True)

In [ ]:
!nohup python belka.py > run.log 2>&1 &  # ~45 min on 24 cores, writes submission.csv

## 2. 1D CNN on raw SMILES bytes

- **Tokenization:** each SMILES string is right-padded to 142 characters, and its raw ASCII bytes are the tokens, embedded with `Embedding(128, d)`. No vocabulary is needed, so no test character can be unknown.
- **Storage:** the full training set is a 14 GB uint8 memmap. The prep streams Arrow record batches out of DuckDB rather than calling `.df()` on 98M strings.
- **Model:** Embedding(128) → Conv1d 32/64/96 (kernel 3) → global max pool → MLP 1024-1024-512 → 3 logits, trained jointly for all three proteins with BCE loss.
- **Training:** fp16 AMP, AdamW, OneCycle LR, batch 4096, 5 epochs over all 98M molecules. Each batch is one numpy fancy-index into the in-RAM array, with no `DataLoader`.
- **Throughput on an RTX 3050:** about 22.7k molecules/s, or 72 minutes per epoch. Doubling the channels (64/128/192) halves throughput.

| Epoch | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| Val AP BRD4 / HSA / sEH | .48/.30/.80 | .52/.35/.84 | .56/.36/.86 | .58/.37/.88 | .59/.38/.88 |

The validation AP above is on a random 0.5% holdout at the natural positive rate.

In [ ]:
%%writefile prep.py
# Full train/test -> raw-ASCII token arrays for the CNN (CPU machine, writes ~14 GB to cnn_data/)
#   tokens.u8   uint8 memmap [N, L]  SMILES bytes, right-padded with ' '
#   labels.u8   uint8 memmap [N, 3]  BRD4, HSA, sEH
#   test_tokens.npy, test_smiles.npy (unique test molecules)
import json, os
import duckdb
import numpy as np

D, OUT = "data", "cnn_data"
os.makedirs(OUT, exist_ok=True)
con = duckdb.connect()
con.execute("SET memory_limit='40GB'")
con.execute(f"CREATE VIEW t AS SELECT * FROM read_parquet('{D}/train.parquet')")

# Train rows come in triples (one per protein) keyed by id // 3 -- verify instead of trusting it.
n, bad, L = con.execute("""
    SELECT count(*), count(*) FILTER (NOT ok), max(len) FROM (
        SELECT min(molecule_smiles) = max(molecule_smiles) AND count(*) = 3
               AND count(DISTINCT protein_name) = 3 AS ok, max(length(molecule_smiles)) AS len
        FROM t GROUP BY id // 3)
""").fetchone()
L_test = con.execute(f"SELECT max(length(molecule_smiles)) FROM read_parquet('{D}/test.parquet')").fetchone()[0]
L = max(L, L_test)
print("molecules", n, "bad groups", bad, "max len", L, flush=True)
assert bad == 0


def tokenize(strings):
    b = np.frombuffer("".join(s.ljust(L) for s in strings).encode("ascii"), np.uint8)
    assert b.max() < 128
    return b.reshape(-1, L)


X =np.memmap(f"{OUT}/tokens.u8", np.uint8, "w+", shape=(n, L))
Y = np.memmap(f"{OUT}/labels.u8", np.uint8, "w+", shape=(n, 3))
reader = con.execute("""
    SELECT any_value(molecule_smiles) AS s,
           max(binds) FILTER (protein_name = 'BRD4')::UTINYINT AS BRD4,
           max(binds) FILTER (protein_name = 'HSA')::UTINYINT AS HSA,
           max(binds) FILTER (protein_name = 'sEH')::UTINYINT AS sEH
    FROM t GROUP BY id // 3 ORDER BY id // 3
""").to_arrow_reader(1_000_000)
i = 0
for batch in reader:
    k = batch.num_rows
    X[i:i + k] = tokenize(batch.column(0).to_pylist())
    Y[i:i + k] = np.stack([batch.column(c).to_numpy() for c in (1, 2, 3)], 1)
    i += k
    if i % 10_000_000 < k:
        print("rows", i, flush=True)
assert i == n
X.flush(); Y.flush()
print("label means", Y.mean(0), flush=True)

test = con.execute(f"SELECT DISTINCT molecule_smiles FROM read_parquet('{D}/test.parquet')").fetchnumpy()["molecule_smiles"]
np.save(f"{OUT}/test_tokens.npy", tokenize(list(test)))
np.save(f"{OUT}/test_smiles.npy", np.array(test, dtype=object), allow_pickle=True)
json.dump({"n": n, "L": L}, open(f"{OUT}/meta.json", "w"))
print("done", flush=True)

In [ ]:
%%writefile train_cnn.py
# Multitask 1D CNN on raw SMILES bytes, all ~98M molecules (GPU machine)
#   nohup python train_cnn.py > cnn.log 2>&1 &
# Writes ckpt_cnn.pt every epoch (resumes from it) and pred_cnn.npy [test molecules, 3].
import json, os, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

D = "cnn_data"
EPOCHS = int(os.environ.get("EPOCHS", 5))
BS = int(os.environ.get("BS", 4096))
LR = float(os.environ.get("LR", 2e-3))
BENCH = int(os.environ.get("BENCH", 0))  # >0: run this many steps, print throughput, exit
CKPT = os.environ.get("CKPT", "ckpt_cnn.pt")

meta = json.load(open(f"{D}/meta.json"))
n, L = meta["n"], meta["L"]
X = np.fromfile(f"{D}/tokens.u8", np.uint8).reshape(n, L)
Y = np.fromfile(f"{D}/labels.u8", np.uint8).reshape(n, 3)
rng = np.random.default_rng(0)
is_va = rng.random(n) < 0.005
va_idx, tr_idx = np.flatnonzero(is_va), np.flatnonzero(~is_va)
print("train", len(tr_idx), "val", len(va_idx), "pos rate", Y.mean(0), flush=True)


class Net(nn.Module):
    def __init__(self, d=128, c=(32, 64, 96), h=1024):  # wider convs: (64,128,192) is 2x slower on a 3050
        super().__init__()
        self.emb = nn.Embedding(128, d)
        chans = (d,) + c
        self.convs = nn.ModuleList(nn.Conv1d(a, b, 3, padding=1) for a, b in zip(chans, chans[1:]))
        self.head = nn.Sequential(nn.Linear(c[-1], h), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(h, h), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(h, h // 2), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h // 2, 3))

    def forward(self, x):
        x = self.emb(x.long()).transpose(1, 2)
        for conv in self.convs:
            x = F.relu(conv(x))
        return self.head(x.amax(2))


def predict(model, toks):
    model.eval()
    out = []
    with torch.no_grad(), torch.autocast("cuda", torch.float16):
        for i in range(0, len(toks), 4096):
            out.append(torch.sigmoid(model(torch.from_numpy(toks[i:i + 4096]).cuda()).float()).cpu().numpy())
    model.train()
    return np.concatenate(out)


def ap(y, p):  # average precision, same definition as sklearn's
    y = y[np.argsort(-p, kind="stable")]
    hits = np.cumsum(y)
    return float((hits / np.arange(1, len(y) + 1))[y == 1].mean())


dev = "cuda"
torch.backends.cudnn.benchmark = True
model = Net().to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
steps_per_epoch = len(tr_idx) // BS
sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=EPOCHS * steps_per_epoch, pct_start=0.05)
scaler = torch.amp.GradScaler()
start = 0
if os.path.exists(CKPT) and not BENCH:
    ck = torch.load(CKPT, weights_only=False)
    for obj, k in ((model, "model"), (opt, "opt"), (sched, "sched"), (scaler, "scaler")):
        obj.load_state_dict(ck[k])
    start = ck["epoch"] + 1
    print("resumed at epoch", start, flush=True)

for epoch in range(start, EPOCHS):
    perm = np.random.default_rng(epoch).permutation(tr_idx)
    t0 = time.time()
    for s in range(steps_per_epoch):
        idx = np.sort(perm[s * BS:(s + 1) * BS])
        x = torch.from_numpy(X[idx]).to(dev, non_blocking=True)
        y = torch.from_numpy(Y[idx]).to(dev, non_blocking=True).float()
        with torch.autocast("cuda", torch.float16):
            loss = F.binary_cross_entropy_with_logits(model(x).float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        if s % 2000 == 0 or (BENCH and s == BENCH):
            el = time.time() - t0
            print(f"ep {epoch} step {s}/{steps_per_epoch} loss {loss.item():.4f} "
                  f"{(s + 1) * BS / el:.0f} mol/s eta {el / (s + 1) * (steps_per_epoch - s) / 60:.0f} min", flush=True)
        if BENCH and s == BENCH:
            raise SystemExit
    pv = predict(model, X[va_idx])
    print(f"epoch {epoch} val AP", [round(ap(Y[va_idx, j], pv[:, j]), 4) for j in range(3)], flush=True)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch}, CKPT)

np.save("pred_cnn.npy", predict(model, np.load(f"{D}/test_tokens.npy")))
print("wrote pred_cnn.npy", flush=True)

In [ ]:
!python prep.py   # ~30 min on the CPU machine, then copy cnn_data/ to the GPU machine
!nohup python train_cnn.py > cnn.log 2>&1 &   # writes ckpt_cnn.pt each epoch (resumable) and pred_cnn.npy

## 3. GNN on molecular graphs

- **Atom features (uint8):** atomic number, degree, formal charge, hydrogen count, aromatic, hybridization, in-ring and chirality.
- **Bond features:** type, conjugated and in-ring.
- **Storage:** graphs go in flat arrays plus per-molecule `atom_ptr` / `bond_ptr` offsets, built with RDKit on 24 cores at about 17k molecules/s.
- **Batch assembly:** fully vectorized with `np.repeat`, about 1.5 ms per 1024-molecule batch. It runs in a producer thread, and the GPU is the bottleneck.
- **Model:** GINE-style message passing written in plain PyTorch with `index_add_`. Hidden size 128, 5 layers, with pre-LayerNorm residual MLPs. The readout is mean pooling plus max pooling, done in fp32, followed by an MLP and 3 logits.
- **Data:** every binder plus 1/4 of the non-binders, 25.7M molecules in total. All 98M graphs would need about 55 GB of RAM.
- **Throughput:** about 11.2k molecules/s, or 37 minutes per epoch, for 5 epochs.

The GNN is the best single model, at 0.247 private. It also differs enough from the CNN that blending the two helps. Its validation AP isn't comparable to the CNN's, because its holdout keeps only a quarter of the non-binders.

In [ ]:
%%writefile gnn_prep.py
# Molecule graphs for the GNN (CPU machine). Writes gnn_data/{train,test}/:
#   atoms.u8 [A, 8]   atomic num, degree, charge+3, Hs, aromatic, hybridization, in ring, chiral tag
#   bonds.u8 [B, 5]   src, dst (molecule-local), type, conjugated, in ring
#   atom_ptr.i64 / bond_ptr.i64 [M+1], train/labels.u8 [M, 3]
# Train = every binder + non-binders with hash % NEG_MOD == 0. Test follows cnn_data/test_smiles.npy order.
import os, sys, time
from multiprocessing import Pool

import duckdb
import numpy as np
from rdkit import Chem, RDLogger

NEG_MOD = int(os.environ.get("NEG_MOD", 4))
BENCH = int(os.environ.get("BENCH", 0))  # >0: featurize this many molecules, print rate, exit
BOND = {Chem.BondType.SINGLE: 0, Chem.BondType.DOUBLE: 1, Chem.BondType.TRIPLE: 2, Chem.BondType.AROMATIC: 3}


def graphs(smiles):
    RDLogger.DisableLog("rdApp.*")
    atoms, bonds, na, nb = [], [], [], []
    for s in smiles:
        m = Chem.MolFromSmiles(s)
        assert m is not None and m.GetNumAtoms() < 256, s
        for a in m.GetAtoms():
            atoms.append((a.GetAtomicNum(), min(a.GetDegree(), 6), a.GetFormalCharge() + 3, min(a.GetTotalNumHs(), 4),
                          a.GetIsAromatic(), int(a.GetHybridization()), a.IsInRing(), int(a.GetChiralTag())))
        for b in m.GetBonds():
            bonds.append((b.GetBeginAtomIdx(), b.GetEndAtomIdx(), BOND.get(b.GetBondType(), 4),
                          b.GetIsConjugated(), b.IsInRing()))
        na.append(m.GetNumAtoms())
        nb.append(m.GetNumBonds())
    return (np.array(atoms, np.uint8).tobytes(), np.array(bonds, np.uint8).tobytes(),
            np.array(na, np.int64), np.array(nb, np.int64))


def write(pool, smiles_batches, out):
    os.makedirs(out, exist_ok=True)
    na, nb = [], []
    with open(f"{out}/atoms.u8", "wb") as fa, open(f"{out}/bonds.u8", "wb") as fb:
        for smiles in smiles_batches:
            chunks = [smiles[i:i + 2000] for i in range(0, len(smiles), 2000)]
            for a, b, la, lb in pool.imap(graphs, chunks):
                fa.write(a); fb.write(b); na.append(la); nb.append(lb)
    for name, lens in (("atom_ptr", na), ("bond_ptr", nb)):
        np.concatenate([[0], np.cumsum(np.concatenate(lens))]).astype(np.int64).tofile(f"{out}/{name}.i64")
    print(out, "molecules", sum(map(len, na)), flush=True)


if __name__ == "__main__":
    con = duckdb.connect()
    con.execute("SET memory_limit='40GB'")
    pool = Pool(os.cpu_count())
    if BENCH:
        smiles = list(np.load("cnn_data/test_smiles.npy", allow_pickle=True)[:BENCH])
        t = time.time()
        write(pool, [smiles], "gnn_bench")
        print(f"{BENCH / (time.time() - t):.0f} mol/s", flush=True)
        sys.exit()

    write(pool, [list(np.load("cnn_data/test_smiles.npy", allow_pickle=True))], "gnn_data/test")

    reader = con.sql(f"""
        SELECT any_value(molecule_smiles) AS s,
               max(binds) FILTER (protein_name = 'BRD4')::UTINYINT AS BRD4,
               max(binds) FILTER (protein_name = 'HSA')::UTINYINT AS HSA,
               max(binds) FILTER (protein_name = 'sEH')::UTINYINT AS sEH
        FROM read_parquet('data/train.parquet') GROUP BY id // 3
        HAVING max(binds) = 1 OR hash(any_value(molecule_smiles)) % {NEG_MOD} = 0
    """).to_arrow_reader(1_000_000)  # id // 3 triples were verified in prep.py
    labels = []

    def batches():
        for batch in reader:
            labels.append(np.stack([batch.column(c).to_numpy() for c in (1, 2, 3)], 1))
            yield batch.column(0).to_pylist()

    write(pool, batches(), "gnn_data/train")
    Y = np.concatenate(labels)
    Y.tofile("gnn_data/train/labels.u8")
    print("labels", Y.shape, Y.mean(0), flush=True)

In [ ]:
%%writefile train_gnn.py
# Multitask GINE-style GNN on molecule graphs from gnn_prep.py (GPU machine)
#   nohup python train_gnn.py > gnn.log 2>&1 &
# Writes ckpt_gnn.pt every epoch (resumes from it) and pred_gnn.npy [test molecules, 3].
import os, queue, threading, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

D = os.environ.get("D", "gnn_data")
EPOCHS = int(os.environ.get("EPOCHS", 5))
BS = int(os.environ.get("BS", 1024))
LR = float(os.environ.get("LR", 1e-3))
BENCH = int(os.environ.get("BENCH", 0))  # >0: run this many steps, print throughput, exit
CKPT = os.environ.get("CKPT", "ckpt_gnn.pt")
SMOKE = os.environ.get("SMOKE") == "1"  # few steps, then predict test (checks the whole path)
ATOM_VOCAB = (128, 7, 8, 5, 2, 8, 2, 4)  # see gnn_prep.py column order
BOND_VOCAB = (5, 2, 2)


class Graphs:
    def __init__(self, d):
        self.atoms = np.fromfile(f"{d}/atoms.u8", np.uint8).reshape(-1, 8)
        self.bonds = np.fromfile(f"{d}/bonds.u8", np.uint8).reshape(-1, 5)
        self.aptr = np.fromfile(f"{d}/atom_ptr.i64", np.int64)
        self.bptr = np.fromfile(f"{d}/bond_ptr.i64", np.int64)
        self.n = len(self.aptr) - 1

    def batch(self, idx):
        la, lb = self.aptr[idx + 1] - self.aptr[idx], self.bptr[idx + 1] - self.bptr[idx]
        off = np.cumsum(la) - la  # each molecule's first atom in the batch
        x = self.atoms[np.repeat(self.aptr[idx] - off, la) + np.arange(la.sum())]
        e = self.bonds[np.repeat(self.bptr[idx] - (np.cumsum(lb) - lb), lb) + np.arange(lb.sum())]
        shift = np.repeat(off, lb)
        src, dst = e[:, 0] + shift, e[:, 1] + shift
        t = lambda a, dt=torch.long: torch.from_numpy(np.ascontiguousarray(a)).to("cuda", dt, non_blocking=True)
        return (t(x), t(np.concatenate([src, dst])), t(np.concatenate([dst, src])),
                t(np.concatenate([e[:, 2:], e[:, 2:]])), t(np.repeat(np.arange(len(idx)), la)), t(la, torch.float))


class Net(nn.Module):
    def __init__(self, h=128, layers=5):
        super().__init__()
        self.atom = nn.ModuleList(nn.Embedding(v, h) for v in ATOM_VOCAB)
        self.bond = nn.ModuleList(nn.ModuleList(nn.Embedding(v, h) for v in BOND_VOCAB) for _ in range(layers))
        self.norm = nn.ModuleList(nn.LayerNorm(h) for _ in range(layers))
        self.mlp = nn.ModuleList(nn.Sequential(nn.Linear(h, 2 * h), nn.ReLU(), nn.Linear(2 * h, h)) for _ in range(layers))
        self.head = nn.Sequential(nn.Linear(2 * h, 512), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, 3))

    def forward(self, x, src, dst, ea, gid, cnt):
        h = sum(emb(x[:, i]) for i, emb in enumerate(self.atom))
        for bond, norm, mlp in zip(self.bond, self.norm, self.mlp):
            msg = F.relu(h[src] + sum(emb(ea[:, i]) for i, emb in enumerate(bond)))
            agg = torch.zeros(h.shape, dtype=msg.dtype, device=h.device).index_add_(0, dst, msg)
            h = h + mlp(norm(h + agg))
        h = h.float()
        g = torch.zeros(len(cnt), h.shape[1], device=h.device)
        mean = g.index_add(0, gid, h) / cnt[:, None]
        mx = g.scatter_reduce(0, gid[:, None].expand_as(h), h, "amax", include_self=False)
        return self.head(torch.cat([mean, mx], 1))


def batches(data, order, bs):  # producer thread: numpy batch assembly overlaps the GPU step
    q = queue.Queue(maxsize=8)

    def work():
        for s in range(0, len(order), bs):
            idx = np.sort(order[s:s + bs])
            q.put((idx, data.batch(idx)))
        q.put(None)
    threading.Thread(target=work, daemon=True).start()
    while (item := q.get()) is not None:
        yield item


def predict(model, data, idx=None):  # idx must be sorted (batches() sorts within each chunk)
    model.eval()
    out = []
    with torch.no_grad(), torch.autocast("cuda", torch.float16):
        for _, b in batches(data, np.arange(data.n) if idx is None else idx, 4096):
            out.append(torch.sigmoid(model(*b).float()).cpu().numpy())
    model.train()
    return np.concatenate(out)


def ap(y, p):  # average precision, same definition as sklearn's
    y = y[np.argsort(-p, kind="stable")]
    hits = np.cumsum(y)
    return float((hits / np.arange(1, len(y) + 1))[y == 1].mean())


if __name__ == "__main__":
    data = Graphs(f"{D}/train")
    Y = np.fromfile(f"{D}/train/labels.u8", np.uint8).reshape(-1, 3)
    assert len(Y) == data.n
    rng = np.random.default_rng(0)
    is_va = rng.random(data.n) < 0.005
    va_idx, tr_idx = np.flatnonzero(is_va), np.flatnonzero(~is_va)
    print("train", len(tr_idx), "val", len(va_idx), "pos rate", Y.mean(0), flush=True)

    model = Net().cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    steps_per_epoch = 200 if SMOKE else len(tr_idx) // BS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=EPOCHS * steps_per_epoch, pct_start=0.05)
    scaler = torch.amp.GradScaler()
    start = 0
    if os.path.exists(CKPT) and not (BENCH or SMOKE):
        ck = torch.load(CKPT, weights_only=False)
        for obj, k in ((model, "model"), (opt, "opt"), (sched, "sched"), (scaler, "scaler")):
            obj.load_state_dict(ck[k])
        start = ck["epoch"] + 1
        print("resumed at epoch", start, flush=True)

    for epoch in range(start, 1 if SMOKE else EPOCHS):
        perm = np.random.default_rng(epoch).permutation(tr_idx)[:steps_per_epoch * BS]
        t0 = time.time()
        for s, (idx, b) in enumerate(batches(data, perm, BS)):
            y = torch.from_numpy(Y[idx]).cuda().float()
            with torch.autocast("cuda", torch.float16):
                loss = F.binary_cross_entropy_with_logits(model(*b).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            if s % 2000 == 0 or (BENCH and s == BENCH):
                el = time.time() - t0
                print(f"ep {epoch} step {s}/{steps_per_epoch} loss {loss.item():.4f} "
                      f"{(s + 1) * BS / el:.0f} mol/s eta {el / (s + 1) * (steps_per_epoch - s) / 60:.0f} min", flush=True)
            if BENCH and s == BENCH:
                raise SystemExit
        pv = predict(model, data, va_idx)
        print(f"epoch {epoch} val AP", [round(ap(Y[va_idx, j], pv[:, j]), 4) for j in range(3)], flush=True)
        if not SMOKE:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "epoch": epoch}, CKPT)

    np.save("pred_gnn.npy", predict(model, Graphs(f"{D}/test")))
    print("wrote pred_gnn.npy", flush=True)

In [ ]:
!python gnn_prep.py   # ~25 min, ~15 GB in gnn_data/
!nohup python train_gnn.py > gnn.log 2>&1 &   # writes ckpt_gnn.pt each epoch (resumable) and pred_gnn.npy

## 4. Blend and submit

The models output probabilities on different scales, and the LightGBM model saw far fewer negatives than the others. The blend therefore rank-averages each model's predictions within each protein. AP only depends on ranking, so averaging ranks loses nothing.

`make_sub.py` also asserts that the submission's IDs match `sample_submission.csv` exactly and that there are no NaNs.

Submit a plain `.csv`. A `.csv.gz` upload fails with *"No .csv or .parquet found after extracting the submission archive"*.

In [ ]:
%%writefile make_sub.py
# Build a submission from one or more prediction sources, rank-averaged per protein (CPU machine)
#   python make_sub.py sub_cnn.csv pred_cnn.npy
#   python make_sub.py sub_blend.csv pred_cnn.npy submission.csv
# .npy sources: [test molecule, 3] aligned to cnn_data/test_smiles.npy; .csv sources: id,binds
import sys
import duckdb
import numpy as np
import pandas as pd

PROTEINS = ["BRD4", "HSA", "sEH"]
out, srcs = sys.argv[1], sys.argv[2:]
test = duckdb.sql("SELECT id, molecule_smiles, protein_name FROM read_parquet('data/test.parquet')").df()
mols = np.load("cnn_data/test_smiles.npy", allow_pickle=True)
row = test.molecule_smiles.map({s: i for i, s in enumerate(mols)}).to_numpy()
col = test.protein_name.map({p: j for j, p in enumerate(PROTEINS)}).to_numpy()


def load(src):
    if src.endswith(".npy"):
        return np.load(src)[row, col]
    return test[["id"]].merge(pd.read_csv(src), on="id", how="left").binds.to_numpy()


preds = [load(s) for s in srcs]
test["binds"] = preds[0] if len(preds) == 1 else np.mean(
    [pd.Series(p).groupby(test.protein_name).rank(pct=True).to_numpy() for p in preds], 0)

sample = pd.read_csv("data/sample_submission.csv", usecols=["id"])
assert len(test) == len(sample) and set(test.id) == set(sample.id), "ids don't match sample_submission"
assert test.binds.notna().all(), "NaN predictions"
test[["id", "binds"]].to_csv(out, index=False)
print("wrote", out, len(test))

In [ ]:
!python make_sub.py sub_cnn_gnn.csv pred_cnn.npy pred_gnn.npy
!kaggle competitions submit -c leash-BELKA -f sub_cnn_gnn.csv -m "CNN + GNN rank average"

## What would improve this next

- **Grouped validation:** split the holdout by building block, so the local score tracks the private leaderboard, which is dominated by unseen building blocks.
- **A bigger GNN on more data:** 192 hidden × 6 layers is about 1.8× slower per epoch, and the GNN's validation AP was still rising at epoch 5.
- **More models in the blend:** extra CNN and GNN runs with different seeds, which is cheap and usually adds a few thousandths.